In [1]:
import numpy as np
from microtool.acquisition_scheme import DiffusionAcquisitionScheme
from microtool.gradient_sampling import sample_uniform_half_sphere

n_directions = 6
n_shells = 3
block_size = n_directions + 1  # 7: b0 + 6 DW directions

shell_bvals = [500, 1500, 2000]
shell_deltas = [0.0258, 0.0205, 0.0264]
shell_Deltas = [0.0434, 0.0331, 0.0365]
initial_te = 0.09

# Build the measurements in shell blocks: b0, then six directions
b_vals_list = []
pulse_durations_list = []
pulse_intervals_list = []
echo_times_list = []

for i in range(n_shells):
    b_vals_list.append(0.0)
    b_vals_list.extend([shell_bvals[i]] * n_directions)

    pulse_durations_list.extend([shell_deltas[i]] * block_size)
    pulse_intervals_list.extend([shell_Deltas[i]] * block_size)
    echo_times_list.extend([initial_te] * block_size)

b_vals = np.array(b_vals_list)
pulse_durations = np.array(pulse_durations_list)
pulse_intervals = np.array(pulse_intervals_list)
echo_times = np.array(echo_times_list)
directions = sample_uniform_half_sphere(len(b_vals))

initial_scheme = DiffusionAcquisitionScheme.from_bvals(
    b_values=b_vals,
    b_vectors=directions,
    pulse_durations=pulse_durations,
    pulse_intervals=pulse_intervals,
    echo_times=echo_times,
)

# 1. TE and timing: repeats every 7 measurements (1 free per shell block of 7)
for name in (
    "EchoTime",
    "DiffusionPulseDuration",
    "DiffusionPulseInterval",
):
    initial_scheme[name].set_repetition_period(block_size)

# 2. Gradient magnitude:
# First fix all b0 measurements so optimize_mask length drops from 21 to 18 (DW only)
b0_mask = b_vals == 0
magnitude = initial_scheme["DiffusionPulseMagnitude"]
magnitude.set_fixed_mask(b0_mask)

# Now among the 18 active DW measurements, sample every 6th measurement (index 0, 6, 12 in the DW subspace)
magnitude.set_repetition_period(n_directions)

print("TE free indices:       ", np.flatnonzero(initial_scheme["EchoTime"].optimize_mask))
print("Timing free indices:   ", np.flatnonzero(initial_scheme["DiffusionPulseDuration"].optimize_mask))
print("Gradient free indices: ", np.flatnonzero(magnitude.optimize_mask))

print(initial_scheme)

TE free indices:        [ 0  7 14]
Timing free indices:    [ 0  7 14]
Gradient free indices:  [ 1  8 15]
Acquisition scheme with 21 measurements and 6 scalar parameters. 
total number of optimized parameters is 12:
  DiffusionPulseMagnitude [mT/mm] in (0.0, 0.065)    DiffusionGradientAnglePhi [rad] (fixed)    DiffusionGradientAngleTheta [rad] (fixed)    DiffusionPulseDuration [s] in (0.001, 0.1)    DiffusionPulseInterval [s] in (0.001, 0.1)    EchoTime [s] in (0.001, 0.1)
-------------------------------------------------  -----------------------------------------  -------------------------------------------  --------------------------------------------  --------------------------------------------  ------------------------------
                                       -0                                         -2.52707                                       1.5708                                          0.0258                                        0.0434                            0.09

In [6]:
from dmipy.core.modeling_framework import MultiCompartmentModel
from dmipy.signal_models import gaussian_models, sphere_models
from microtool.dmipy import DmipyMultiTissueModel

# Sphere: radius in meters, diffusivity in m²/s
sphere = sphere_models.S4SphereGaussianPhaseApproximation(diffusion_constant=2.0e-9, diameter=12e-6)
ball = gaussian_models.G1Ball(lambda_iso=2.0e-9)

dmipy_raw = MultiCompartmentModel([sphere, ball])
dmipy_model = DmipyMultiTissueModel(
    dmipy_raw,
    volume_fractions=[0.60, 0.40],
)
dmipy_model["G1Ball_1_lambda_iso"].optimize = False
print(dmipy_model)
dmipy_signal = dmipy_model(initial_scheme)
dmipy_jacobian = dmipy_model.scaled_jacobian(initial_scheme)

print("Dmipy signal:", dmipy_signal)
print("Signal shape:", dmipy_signal.shape)
print("Dmipy Jacobian:")
print(dmipy_jacobian)
print("Jacobian shape:", dmipy_jacobian.shape)


Tissue model with 5 scalar parameters:
Tissue-parameter                                 Value    Scale  Optimize    Fit    Fit Bounds
---------------------------------------------  -------  -------  ----------  -----  ------------------------------------------------
S4SphereGaussianPhaseApproximation_1_diameter  1.2e-05    1e-06  True        True   (1e-08, 1.9999999999999998e-05)
G1Ball_1_lambda_iso                            2e-09      1e-09  False       True   (1.0000000000000002e-10, 3.0000000000000004e-09)
partial_volume_0                               0.6        1      True        True   (0.0, 1.0)
partial_volume_1                               0.4        1      True        True   (0.0, 1.0)
S0                                             1          1      False       False  (0.0, inf)
Dmipy signal: [1.         0.73078388 0.73078388 0.73078388 0.73078388 0.73078388
 0.73078388 1.         0.54620509 0.54620509 0.54620509 0.54620509
 0.54620509 0.54620509 1.         0.5312988  0.5312